# Res-UNet 2.5D Segmentation — Inference Notebook (Colab)

Inferensi Residual U-Net 2.5D pada volume µCT rock core. Arsitektur dan konvensi **persis** mengikuti training notebook (`resunet_segmentation_colab_2p5D.ipynb`) — `FILTERS=[32,64,128,256,512]`, decoder bilinear `Upsample`, output 2 kelas softmax dengan **class index 0 = pori**.

**Pipeline:**
1. Load checkpoint (`model_state`) + norm_stats hasil training
2. Load volume input (TIFF 3D), normalisasi p1-p99 (adaptive atau dari norm_stats)
3. Inferensi 2.5D patch-based + Hanning blending → peta probabilitas pori `softmax(logits)[:, 0]`
4. **Dua jalur biner:** (a) thresh=0.5, (b) Otsu(prob) — apple-to-apple dgn RED-CNN
5. Metrik vs GT (kalau ada): PSNR/SSIM/RMSE prob-vs-GT + Dice/IoU biner
6. Petrofisika (porositas, SSA via boundary-erosion, Kozeny-Carman dalam mD)
7. Visualisasi & save outputs

**Konvensi penting:**
- Input: 5-channel 2.5D (offset -2..+2 dari slice tengah), mirror-clamp di tepi
- GT file: pori = 1 (binary {0,1}); prediksi biner juga pori = 1 (setelah threshold)
- Voxel 5.714 µm; τ = 2.5; k₀ = 2.0


## 0 · Setup Colab — Mount Drive & Install Dependencies

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install dependencies (torch sudah ada di Colab)
!pip install -q tifffile scikit-image scipy psutil

# Cek GPU aktif
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 1 · Import & Device Setup

In [ ]:
import os, gc, time, json, warnings
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from datetime import datetime
import multiprocessing

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

from tifffile import imread, imwrite
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_erosion
import psutil

warnings.filterwarnings('ignore')

CPU_COUNT = multiprocessing.cpu_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("WARNING: CUDA tidak tersedia, inference akan lambat.")
print(f"PyTorch: {torch.__version__}  |  CPU cores: {CPU_COUNT}")


## 2 · Konfigurasi

**Edit di cell ini saja untuk run normal.** Cell-cell lain di-parameterized.

In [ ]:
class InfConfig:
    # ── Path checkpoint hasil training ─────────────────────────────────────
    MODEL_PATH = '/content/drive/MyDrive/Dataset TA/Res-Net_Seg_2/resunet_seg_best.pth'

    # ── Path volume INPUT (TIFF 3D, grayscale noisy, mis. 0.8° atau 0.4°) ──
    INPUT_PATH = '/content/drive/MyDrive/Dataset TA/Libo/.../poor_08_volume.tif'

    # ── Path GT segmentasi biner (pori=1). Set None kalau tidak punya. ─────
    GT_SEG_PATH = '/content/drive/MyDrive/Dataset TA/Libo/.../ground_truth_seg.tif'

    # ── Output directory ───────────────────────────────────────────────────
    OUT_DIR = '/content/drive/MyDrive/Dataset TA/Res-Net_Seg_2/inference_run1'

    # ── Cropping (None = pakai full volume) ────────────────────────────────
    CROP_SIZE   = None       # int atau None
    CROP_ORIGIN = None       # (x0,y0,z0) atau None (otomatis center crop)

    # ── Normalisasi ────────────────────────────────────────────────────────
    # 'adaptive' : p1/p99 dihitung dari volume input ini (default, konsisten dgn
    #              pipeline cross-dataset)
    # 'training' : pakai p1/p99 dari norm_stats checkpoint (pilih sumber dgn
    #              NORM_SOURCE di bawah). Cocok kalau volume input ini SAMA
    #              dengan yang dipakai training.
    NORM_MODE   = 'adaptive'
    NORM_SOURCE = '0.8'      # '0.8' atau '0.4' — dipakai hanya bila NORM_MODE='training'

    # ── Arsitektur (HARUS identik training) ────────────────────────────────
    UNET_IN_CH = 5           # 2.5D 5-channel
    N_CLASSES  = 2           # softmax 2 kelas; class index 0 = PORI

    # ── Inference patching (identik training) ──────────────────────────────
    PATCH_SIZE  = 256
    STRIDE_TEST = 96
    USE_AMP     = True       # Colab GPU → ON
    BATCH_SIZE  = 8          # patches per forward pass (turunkan ke 4 kalau OOM)

    # ── Block length untuk clamp jendela 2.5D ──────────────────────────────
    # Default None = volume tunggal, jendela 2.5D clamp ke [0, Z-1].
    # Set ke panjang sumber jika input adalah concat beberapa volume.
    BLOCK_LEN = None

    # ── Petrofisika ────────────────────────────────────────────────────────
    VOXEL_SIZE_UM    = 5.714
    KC_TORTUOSITY    = 2.5
    KC_SHAPE_FACTOR  = 2.0

    # ── Output options ─────────────────────────────────────────────────────
    SAVE_PROBMAP     = True
    SAVE_BIN_THR05   = True
    SAVE_BIN_OTSU    = True

cfg = InfConfig()
os.makedirs(cfg.OUT_DIR, exist_ok=True)
PORE_CLASS_IDX = 0   # konvensi training: class 0 = pori

print("="*65); print("  CONFIG"); print("="*65)
print(f"  Model     : {cfg.MODEL_PATH}")
print(f"  Input     : {cfg.INPUT_PATH}")
print(f"  GT seg    : {cfg.GT_SEG_PATH}")
print(f"  Output    : {cfg.OUT_DIR}")
print(f"  Norm mode : {cfg.NORM_MODE}" + (f" (source={cfg.NORM_SOURCE})" if cfg.NORM_MODE=='training' else ""))
print(f"  Patch     : {cfg.PATCH_SIZE}²  stride={cfg.STRIDE_TEST}  AMP={cfg.USE_AMP}")


## 3 · Arsitektur Residual U-Net 2.5D

**Disalin PERSIS dari training notebook** — `FILTERS=[32,64,128,256,512]`, encoder dgn `MaxPool2d`, decoder dgn `nn.Upsample(bilinear)`. State-dict keys harus match 100%.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                       nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = ResidualBlock(in_ch, out_ch)
        self.pool  = nn.MaxPool2d(2, 2)

    def forward(self, x):
        skip = self.block(x)
        return self.pool(skip), skip


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.block = ResidualBlock(in_ch + skip_ch, out_ch)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.block(torch.cat([x, skip], dim=1))


class ResidualUNet(nn.Module):
    """Residual U-Net 2.5D. in_ch=5, output 2 kelas (logits). class 0 = pori."""
    FILTERS = [32, 64, 128, 256, 512]

    def __init__(self, in_ch=5, n_classes=2):
        super().__init__()
        f = self.FILTERS
        self.enc1 = EncoderBlock(in_ch, f[0]); self.enc2 = EncoderBlock(f[0], f[1])
        self.enc3 = EncoderBlock(f[1], f[2]);  self.enc4 = EncoderBlock(f[2], f[3])
        self.bottleneck = ResidualBlock(f[3], f[4])
        self.dec4 = DecoderBlock(f[4], f[3], f[3]); self.dec3 = DecoderBlock(f[3], f[2], f[2])
        self.dec2 = DecoderBlock(f[2], f[1], f[1]); self.dec1 = DecoderBlock(f[1], f[0], f[0])
        self.out  = nn.Conv2d(f[0], n_classes, kernel_size=1)

    def forward(self, x):
        x, s1 = self.enc1(x); x, s2 = self.enc2(x)
        x, s3 = self.enc3(x); x, s4 = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4); x = self.dec3(x, s3)
        x = self.dec2(x, s2); x = self.dec1(x, s1)
        return self.out(x)            # logits (B, 2, H, W)


def count_parameters(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

# Sanity: bangun dummy & verifikasi shape
_m = ResidualUNet(in_ch=cfg.UNET_IN_CH, n_classes=cfg.N_CLASSES).to(device)
with torch.no_grad():
    _o = _m(torch.randn(2, cfg.UNET_IN_CH, 256, 256, device=device))
assert _o.shape == (2, cfg.N_CLASSES, 256, 256), f"shape mismatch {_o.shape}"
print(f"Res-UNet 2.5D OK — params: {count_parameters(_m):,}  out={tuple(_o.shape)}")
del _m, _o
torch.cuda.empty_cache()


## 4 · Load Checkpoint

Training menyimpan dict dengan key `'model_state'`, `'epoch'`, `'val_loss'`, `'val_dice'`, `'config'`, `'norm_stats'`.

In [ ]:
print("Loading checkpoint ...")
ckpt = torch.load(cfg.MODEL_PATH, map_location=device, weights_only=False)

model = ResidualUNet(in_ch=cfg.UNET_IN_CH, n_classes=cfg.N_CLASSES).to(device)

if 'model_state' in ckpt:
    sd = ckpt['model_state']
elif 'model_state_dict' in ckpt:
    sd = ckpt['model_state_dict']
elif 'state_dict' in ckpt:
    sd = ckpt['state_dict']
else:
    sd = ckpt   # raw state_dict

# Strip 'module.' prefix kalau training pakai DataParallel
sd = {k.replace('module.', ''): v for k, v in sd.items()}

missing, unexpected = model.load_state_dict(sd, strict=False)
if missing:
    print(f"[WARN] Missing keys ({len(missing)}):")
    for k in missing[:8]: print(f"   {k}")
if unexpected:
    print(f"[WARN] Unexpected keys ({len(unexpected)}):")
    for k in unexpected[:8]: print(f"   {k}")
if not missing and not unexpected:
    print("✓ State dict match perfect")

model.eval()

# Info checkpoint
for k in ('epoch', 'val_loss', 'val_dice'):
    if k in ckpt: print(f"  ckpt[{k!r}] = {ckpt[k]}")

# Norm stats dari training
norm_stats_ckpt = ckpt.get('norm_stats', None)
if norm_stats_ckpt is not None:
    if hasattr(norm_stats_ckpt, 'item'): norm_stats_ckpt = norm_stats_ckpt.item()
    print(f"  norm_stats keys: {list(norm_stats_ckpt.keys())}")
else:
    print("  norm_stats: tidak ada di checkpoint")

print(f"\nModel siap pada {device}")


## 5 · Load Volume Input & Normalisasi

Mengikuti `normalize_volume()` dari training: clip ke [p1, p99] lalu rescale ke [0,1].

In [ ]:
def load_and_crop_volume(path, crop_size=None, crop_origin=None):
    print(f"  Loading: {os.path.basename(path)} ...", end='', flush=True)
    vol = imread(path).astype(np.float32)
    print(f" shape={vol.shape}")
    if crop_size is not None:
        D, H, W = vol.shape; cs = crop_size
        if crop_origin is not None: x0, y0, z0 = crop_origin
        else: x0 = max(0,(W-cs)//2); y0 = max(0,(H-cs)//2); z0 = max(0,(D-cs)//2)
        x0 = min(x0,W-cs); y0 = min(y0,H-cs); z0 = min(z0,D-cs)
        vol = vol[z0:z0+cs, y0:y0+cs, x0:x0+cs]
        print(f"  → Cropped → {vol.shape}")
    return vol


def normalize_volume(vol, p1=None, p99=None):
    """Identik training: clip 1-99 percentile, lalu rescale ke [0,1]."""
    if p1  is None: p1  = float(np.percentile(vol, 1))
    if p99 is None: p99 = float(np.percentile(vol, 99))
    vol_n = np.clip((vol - p1) / (p99 - p1 + 1e-8), 0.0, 1.0)
    return vol_n.astype(np.float32), p1, p99


t0 = time.time()
vol_raw = load_and_crop_volume(cfg.INPUT_PATH, cfg.CROP_SIZE, cfg.CROP_ORIGIN)
print(f"  → loaded in {time.time()-t0:.1f}s, dtype={vol_raw.dtype}, "
      f"min={vol_raw.min():.2f}, max={vol_raw.max():.2f}")

# Pilih normalisasi
if cfg.NORM_MODE == 'training' and norm_stats_ckpt is not None:
    p1_key  = f'p1_{cfg.NORM_SOURCE}'
    p99_key = f'p99_{cfg.NORM_SOURCE}'
    if p1_key in norm_stats_ckpt and p99_key in norm_stats_ckpt:
        p1, p99 = float(norm_stats_ckpt[p1_key]), float(norm_stats_ckpt[p99_key])
        vol_norm, _, _ = normalize_volume(vol_raw, p1, p99)
        print(f"  Normalisasi: TRAINING stats ({cfg.NORM_SOURCE})  p1={p1:.2f}  p99={p99:.2f}")
    else:
        print(f"  [WARN] {p1_key}/{p99_key} tidak ada di norm_stats; fallback ke adaptive")
        vol_norm, p1, p99 = normalize_volume(vol_raw)
        print(f"  Normalisasi: ADAPTIVE  p1={p1:.2f}  p99={p99:.2f}")
else:
    vol_norm, p1, p99 = normalize_volume(vol_raw)
    print(f"  Normalisasi: ADAPTIVE  p1={p1:.2f}  p99={p99:.2f}")

Z, H, W = vol_norm.shape
print(f"  Final volume: {vol_norm.shape}  range [{vol_norm.min():.3f}, {vol_norm.max():.3f}]")
del vol_raw
gc.collect()


## 6 · Load Ground Truth Segmentasi (Opsional)

In [ ]:
seg_gt = None
if cfg.GT_SEG_PATH and os.path.exists(cfg.GT_SEG_PATH):
    seg_raw = load_and_crop_volume(cfg.GT_SEG_PATH, cfg.CROP_SIZE, cfg.CROP_ORIGIN)
    seg_gt = (seg_raw > 0.5).astype(np.float32)
    print(f"  GT loaded: shape={seg_gt.shape}, unique={np.unique(seg_gt)}")
    print(f"  GT porositas total = {seg_gt.mean()*100:.3f} %")
    assert seg_gt.shape == vol_norm.shape, \
        f"Shape mismatch: GT {seg_gt.shape} vs Input {vol_norm.shape}"
    del seg_raw
else:
    print("  GT tidak tersedia — skip metrik & petrofisika GT.")


## 7 · Inference Helpers — 2.5D Patch + Hanning Blending

Identik training, tapi dibuat **batched** untuk speed di GPU.

In [ ]:
def stack_2p5d(vol, z, block_len=None):
    """5-channel jendela 2.5D di sekitar z (clamp dalam blok)."""
    if block_len is None:
        lo, hi = 0, vol.shape[0] - 1
    else:
        b = z // block_len
        lo, hi = b * block_len, (b + 1) * block_len - 1
    idxs = [min(max(z + off, lo), hi) for off in (-2, -1, 0, 1, 2)]
    return np.stack([vol[i] for i in idxs], axis=0)   # (5, H, W)


def get_patch_coords(H, W, patch, stride):
    ys = list(range(0, H - patch + 1, stride))
    if not ys or ys[-1] + patch < H: ys.append(max(0, H - patch))
    xs = list(range(0, W - patch + 1, stride))
    if not xs or xs[-1] + patch < W: xs.append(max(0, W - patch))
    # dedupe sambil pertahankan urutan
    ys = sorted(set(ys)); xs = sorted(set(xs))
    return [(y, x) for y in ys for x in xs]


@torch.no_grad()
def infer_slice(win5, model, cfg):
    """
    win5: (5, H, W) jendela 2.5D ternormalisasi.
    Return: peta probabilitas pori (H, W) — softmax(logits)[class 0].
    """
    _, H, W = win5.shape
    ps, st = cfg.PATCH_SIZE, cfg.STRIDE_TEST

    # Pad reflect kalau lebih kecil dari patch
    pad_h = max(0, ps - H); pad_w = max(0, ps - W)
    if pad_h or pad_w:
        win5 = np.pad(win5, ((0,0),(0,pad_h),(0,pad_w)), mode='reflect')
    Hp, Wp = win5.shape[1], win5.shape[2]

    coords = get_patch_coords(Hp, Wp, ps, st)
    g = (np.outer(np.hanning(ps), np.hanning(ps)) + 1e-6).astype(np.float32)

    prob_sum = np.zeros((Hp, Wp), dtype=np.float64)
    wgt_sum  = np.zeros((Hp, Wp), dtype=np.float64)

    for b0 in range(0, len(coords), cfg.BATCH_SIZE):
        batch_coords = coords[b0:b0 + cfg.BATCH_SIZE]
        patches = np.stack([win5[:, y:y+ps, x:x+ps] for (y,x) in batch_coords])  # (B,5,ps,ps)
        t = torch.from_numpy(patches).float().to(device, non_blocking=True)
        with autocast(enabled=cfg.USE_AMP):
            logits = model(t)
            p_pore = torch.softmax(logits, dim=1)[:, PORE_CLASS_IDX].cpu().float().numpy()
        for i, (y, x) in enumerate(batch_coords):
            prob_sum[y:y+ps, x:x+ps] += p_pore[i] * g
            wgt_sum [y:y+ps, x:x+ps] += g

    prob = np.clip(prob_sum / wgt_sum, 0.0, 1.0).astype(np.float32)
    return prob[:H, :W]   # crop kembali ke ukuran asli


print("Inference helpers loaded.")


## 8 · Run Inference — Full Volume

In [ ]:
prob_vol = np.zeros((Z, H, W), dtype=np.float32)

t0 = time.time()
for z in tqdm(range(Z), desc="Inferring slices"):
    win5 = stack_2p5d(vol_norm, z, block_len=cfg.BLOCK_LEN)
    prob_vol[z] = infer_slice(win5, model, cfg)

dt = time.time() - t0
print(f"\nInference done in {dt:.1f}s ({dt/Z*1000:.1f} ms/slice)")
print(f"Probability map: shape={prob_vol.shape}, range=[{prob_vol.min():.4f}, {prob_vol.max():.4f}]")
print(f"Mean pore probability: {prob_vol.mean():.4f}")


## 9 · Dua Jalur Biner — thr=0.5 vs Otsu(prob)

Persis pipeline training (apple-to-apple dgn RED-CNN).

In [ ]:
# (1) Segmentasi langsung: prob >= 0.5
bin_05 = (prob_vol >= 0.5).astype(np.uint8)

# (2) Otsu pada peta probabilitas (apple-to-apple dengan RED-CNN→Otsu)
otsu_t = float(threshold_otsu(prob_vol.flatten()))
bin_otsu = (prob_vol >= otsu_t).astype(np.uint8)

print(f"  thr=0.5  : porositas = {bin_05.mean()*100:.3f} %")
print(f"  Otsu(prob)={otsu_t:.4f}  porositas = {bin_otsu.mean()*100:.3f} %")


## 10 · Metrik vs Ground Truth (kalau GT tersedia)

In [ ]:
def seg_metrics(pred_bin, gt_bin):
    p = pred_bin.astype(bool); t = gt_bin.astype(bool)
    tp = np.logical_and(p, t).sum()
    fp = np.logical_and(p, ~t).sum()
    fn = np.logical_and(~p, t).sum()
    tn = np.logical_and(~p, ~t).sum()
    return {
        'Dice': float(2*tp/(2*tp+fp+fn+1e-8)),
        'IoU' : float(tp/(tp+fp+fn+1e-8)),
        'Acc' : float((tp+tn)/(tp+tn+fp+fn+1e-8)),
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
    }


reg_metrics = None
seg_m_05 = seg_m_otsu = None

if seg_gt is not None:
    # Regression-style metrics (prob_vol vs GT biner) — per slice lalu rata-rata
    psnr_list, ssim_list, rmse_list = [], [], []
    for i in tqdm(range(Z), desc="PSNR/SSIM per slice"):
        psnr_list.append(psnr_fn(seg_gt[i], prob_vol[i], data_range=1.0))
        ssim_list.append(ssim_fn(seg_gt[i], prob_vol[i], data_range=1.0))
        rmse_list.append(float(np.sqrt(np.mean((seg_gt[i] - prob_vol[i])**2))))
    reg_metrics = {
        'PSNR_mean': float(np.mean(psnr_list)), 'PSNR_std': float(np.std(psnr_list)),
        'SSIM_mean': float(np.mean(ssim_list)), 'SSIM_std': float(np.std(ssim_list)),
        'RMSE_mean': float(np.mean(rmse_list)), 'RMSE_std': float(np.std(rmse_list)),
    }

    seg_m_05   = seg_metrics(bin_05,   seg_gt)
    seg_m_otsu = seg_metrics(bin_otsu, seg_gt)

    print("\n=== Regression-style metrics (prob vs GT biner) ===")
    print(f"  PSNR : {reg_metrics['PSNR_mean']:.4f} ± {reg_metrics['PSNR_std']:.4f} dB")
    print(f"  SSIM : {reg_metrics['SSIM_mean']:.4f} ± {reg_metrics['SSIM_std']:.4f}")
    print(f"  RMSE : {reg_metrics['RMSE_mean']:.6f} ± {reg_metrics['RMSE_std']:.6f}")

    print("\n=== Segmentation metrics ===")
    print(f"  [thr=0.5 ]  Dice={seg_m_05['Dice']:.4f}  "
          f"IoU={seg_m_05['IoU']:.4f}  Acc={seg_m_05['Acc']:.4f}")
    print(f"  [Otsu    ]  Dice={seg_m_otsu['Dice']:.4f}  "
          f"IoU={seg_m_otsu['IoU']:.4f}  Acc={seg_m_otsu['Acc']:.4f}")
else:
    print("  GT tidak ada — skip metrik.")


## 11 · Analisis Petrofisika (Porositas, SSA, Kozeny-Carman)

Identik training: SSA via boundary-erosion (`bin - binary_erosion(bin)`), Kozeny-Carman:

$$k = \frac{\phi^3}{k_0 \cdot \tau^2 \cdot S_v^2 \cdot (1-\phi)^2}$$


In [ ]:
def calculate_porosity(bin_vol):
    per_sl = bin_vol.mean(axis=(1, 2))
    return {
        'porosity_total': float(bin_vol.mean()),
        'porosity_percent': float(bin_vol.mean() * 100),
        'pore_voxels': int(bin_vol.sum()),
        'total_voxels': int(bin_vol.size),
        'porosity_per_slice': per_sl,
    }


def calculate_ssa_volume(bin_vol, voxel_um):
    """SSA 3D (1/m): luas permukaan pori (boundary erosion) / volume total."""
    vox_m = voxel_um * 1e-6
    D, Hh, Ww = bin_vol.shape
    vol_m3 = D * Hh * Ww * vox_m**3
    surface = bin_vol.astype(np.int32) - binary_erosion(bin_vol.astype(bool)).astype(np.int32)
    sa_m2 = int(surface.sum()) * vox_m**2
    Sv = sa_m2 / vol_m3 if vol_m3 > 0 else 0.0
    return {'surface_voxels': int(surface.sum()),
            'SSA_per_m': float(Sv),
            'SSA_m2_per_cm3': float(Sv * 1e-2)}  # 1/m → m²/cm³ = ×1e-2


def kozeny_carman(phi, Sv_m, tau, k0):
    if phi <= 0 or phi >= 1 or Sv_m <= 0:
        return {'K_m2': float('nan'), 'K_mD': float('nan')}
    K_m2 = phi**3 / (k0 * tau**2 * Sv_m**2 * (1 - phi)**2)
    return {'K_m2': float(K_m2), 'K_mD': float(K_m2 / 9.869233e-16)}


def petro_summary(label, bin_vol, cfg):
    por = calculate_porosity(bin_vol)
    ssa = calculate_ssa_volume(bin_vol, cfg.VOXEL_SIZE_UM)
    kc  = kozeny_carman(por['porosity_total'], ssa['SSA_per_m'],
                        cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR)
    print(f"  [{label:8s}]  φ = {por['porosity_percent']:7.3f}%   "
          f"SSA = {ssa['SSA_per_m']:.3e} 1/m   "
          f"k = {kc['K_mD']:.3f} mD")
    return {'porosity': por, 'ssa': ssa, 'kc': kc}


print("="*65); print("  PETROPHYSICS"); print("="*65)
petro_05   = petro_summary("thr=0.5", bin_05,   cfg)
petro_otsu = petro_summary("Otsu",    bin_otsu, cfg)
petro_gt   = petro_summary("GT",      seg_gt.astype(np.uint8), cfg) if seg_gt is not None else None

if petro_gt is not None:
    print("\n  Relative error vs GT:")
    for label, p in (('thr=0.5', petro_05), ('Otsu', petro_otsu)):
        dphi = (p['porosity']['porosity_percent'] - petro_gt['porosity']['porosity_percent']) / petro_gt['porosity']['porosity_percent'] * 100
        dssa = (p['ssa']['SSA_per_m'] - petro_gt['ssa']['SSA_per_m']) / petro_gt['ssa']['SSA_per_m'] * 100 if petro_gt['ssa']['SSA_per_m']>0 else float('nan')
        dk = (p['kc']['K_mD'] - petro_gt['kc']['K_mD']) / petro_gt['kc']['K_mD'] * 100 if petro_gt['kc']['K_mD']>0 else float('nan')
        print(f"    [{label:8s}]  Δφ = {dphi:+6.2f}%   ΔSSA = {dssa:+6.2f}%   Δk = {dk:+6.2f}%")


## 12 · Permeabilitas Per-Slice (Paralel)

Mirror dari `run_petrophysics_parallel()` di training, untuk grafik porositas/permeabilitas vs Z.

In [ ]:
def _process_single_slice(args):
    z_idx, bin_sl, phi_frac, voxel_um, tortuosity, shape_factor = args
    vox_m = voxel_um*1e-6; Hh, Ww = bin_sl.shape
    surf = bin_sl.astype(np.int32) - binary_erosion(bin_sl.astype(bool)).astype(np.int32)
    sa_m2 = int(surf.sum())*vox_m**2; vol_m3 = Hh*Ww*vox_m**3
    Sv_m = sa_m2/vol_m3 if vol_m3 > 0 else 0.0
    if 0 < phi_frac < 1 and Sv_m > 0:
        K_m2 = phi_frac**3 / (shape_factor*tortuosity**2*Sv_m**2*(1-phi_frac)**2)
        K_mD = K_m2 / 9.869233e-16
    else:
        K_mD = float('nan')
    return z_idx, float(Sv_m), float(K_mD)


def run_petro_per_slice(bin_vol, label):
    n = bin_vol.shape[0]
    phi_per_slice = bin_vol.mean(axis=(1, 2))
    args_list = [(z, bin_vol[z], float(phi_per_slice[z]),
                  cfg.VOXEL_SIZE_UM, cfg.KC_TORTUOSITY, cfg.KC_SHAPE_FACTOR)
                 for z in range(n)]
    Sv_arr = np.zeros(n); K_arr = np.full(n, float('nan'))
    n_workers = max(1, CPU_COUNT - 1)
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_process_single_slice, a): a[0] for a in args_list}
        for fut in tqdm(as_completed(futures), total=n, desc=f'Petro [{label}]'):
            z, Sv, K = fut.result(); Sv_arr[z] = Sv; K_arr[z] = K
    return phi_per_slice, Sv_arr, K_arr


phi_05,   Sv_05,   K_05   = run_petro_per_slice(bin_05,   "thr=0.5")
phi_otsu, Sv_otsu, K_otsu = run_petro_per_slice(bin_otsu, "Otsu")
phi_gt = Sv_gt = K_gt = None
if seg_gt is not None:
    phi_gt, Sv_gt, K_gt = run_petro_per_slice(seg_gt.astype(np.uint8), "GT")

print(f"\n  Mean K [thr=0.5] : {np.nanmean(K_05):.3f} mD")
print(f"  Mean K [Otsu]    : {np.nanmean(K_otsu):.3f} mD")
if K_gt is not None:
    print(f"  Mean K [GT]      : {np.nanmean(K_gt):.3f} mD")


## 13 · Visualisasi

In [ ]:
# Panel slice tengah
mid = Z // 2
n_panels = 5 if seg_gt is not None else 4
fig, axes = plt.subplots(1, n_panels, figsize=(5*n_panels, 5))
fig.suptitle(f'Res-UNet 2.5D Inference — slice tengah (z={mid})', fontsize=12, fontweight='bold')

axes[0].imshow(vol_norm[mid], cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Input (normalized)'); axes[0].axis('off')

im = axes[1].imshow(prob_vol[mid], cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Prob pori (softmax[0])'); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

axes[2].imshow(bin_05[mid], cmap='binary_r')
axes[2].set_title('Biner thr=0.5  (pore=white)'); axes[2].axis('off')

axes[3].imshow(bin_otsu[mid], cmap='binary_r')
axes[3].set_title(f'Biner Otsu (t={otsu_t:.3f})'); axes[3].axis('off')

if seg_gt is not None:
    axes[4].imshow(seg_gt[mid], cmap='binary_r')
    axes[4].set_title('GT (pore=white)'); axes[4].axis('off')

plt.tight_layout()
panel_path = os.path.join(cfg.OUT_DIR, 'panel_segmentation.png')
plt.savefig(panel_path, dpi=150, bbox_inches='tight'); plt.show()
print(f"  Saved: {panel_path}")


In [ ]:
# Porositas & permeabilitas per slice
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sl = np.arange(Z)

if phi_gt is not None:
    axes[0].plot(phi_gt*100,   sl, color='black',    lw=1.5, label='GT')
axes[0].plot(phi_05*100,   sl, color='royalblue', lw=1.0, label='thr=0.5')
axes[0].plot(phi_otsu*100, sl, color='tomato',    lw=1.0, label='Otsu')
axes[0].set_xlabel('Porositas (%)'); axes[0].set_ylabel('Slice (Z)')
axes[0].set_title('Porositas per-slice'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].invert_yaxis()

if K_gt is not None:
    axes[1].plot(K_gt,   sl, color='black',    lw=1.5, label='GT')
axes[1].plot(K_05,   sl, color='royalblue', lw=1.0, label='thr=0.5')
axes[1].plot(K_otsu, sl, color='tomato',    lw=1.0, label='Otsu')
axes[1].set_xlabel('Permeabilitas (mD)'); axes[1].set_ylabel('Slice (Z)')
axes[1].set_title('Permeabilitas K-C per-slice'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_xscale('symlog'); axes[1].invert_yaxis()

plt.tight_layout()
petro_plot_path = os.path.join(cfg.OUT_DIR, 'panel_petrophysics.png')
plt.savefig(petro_plot_path, dpi=150, bbox_inches='tight'); plt.show()
print(f"  Saved: {petro_plot_path}")


## 14 · Save Outputs

In [ ]:
print("="*65); print("  SAVING OUTPUTS"); print("="*65)

if cfg.SAVE_PROBMAP:
    p = os.path.join(cfg.OUT_DIR, 'prob_pore.tif')
    imwrite(p, prob_vol)
    print(f"  ✓ {p}  ({os.path.getsize(p)/1e6:.1f} MB)")

if cfg.SAVE_BIN_THR05:
    p = os.path.join(cfg.OUT_DIR, 'seg_thr05.tif')
    imwrite(p, (bin_05*255).astype(np.uint8))
    print(f"  ✓ {p}")

if cfg.SAVE_BIN_OTSU:
    p = os.path.join(cfg.OUT_DIR, 'seg_otsu.tif')
    imwrite(p, (bin_otsu*255).astype(np.uint8))
    print(f"  ✓ {p}")

# CSV per-slice
df_dict = {
    'slice_z': np.arange(Z),
    'Porosity_thr05_%': phi_05*100,
    'Porosity_otsu_%' : phi_otsu*100,
    'Perm_thr05_mD'   : K_05,
    'Perm_otsu_mD'    : K_otsu,
    'SSA_thr05_per_m' : Sv_05,
    'SSA_otsu_per_m'  : Sv_otsu,
}
if seg_gt is not None:
    df_dict.update({
        'Porosity_GT_%' : phi_gt*100,
        'Perm_GT_mD'    : K_gt,
        'SSA_GT_per_m'  : Sv_gt,
        'PSNR_dB'       : psnr_list,
        'SSIM'          : ssim_list,
        'RMSE'          : rmse_list,
    })
df = pd.DataFrame(df_dict)
csv_path = os.path.join(cfg.OUT_DIR, 'metrics_per_slice.csv')
df.to_csv(csv_path, index=False)
print(f"  ✓ {csv_path}")

# Summary JSON
summary = {
    'model_path': cfg.MODEL_PATH,
    'input_path': cfg.INPUT_PATH,
    'gt_path': cfg.GT_SEG_PATH,
    'timestamp': datetime.now().isoformat(),
    'volume_shape': list(prob_vol.shape),
    'normalization': {'mode': cfg.NORM_MODE, 'p1': float(p1), 'p99': float(p99)},
    'thresholds': {'fixed': 0.5, 'otsu': otsu_t},
    'porosity_%': {
        'thr0.5': petro_05['porosity']['porosity_percent'],
        'otsu'  : petro_otsu['porosity']['porosity_percent'],
    },
    'SSA_per_m': {
        'thr0.5': petro_05['ssa']['SSA_per_m'],
        'otsu'  : petro_otsu['ssa']['SSA_per_m'],
    },
    'Permeability_mD_KC': {
        'thr0.5': petro_05['kc']['K_mD'],
        'otsu'  : petro_otsu['kc']['K_mD'],
    },
}
if petro_gt is not None:
    summary['porosity_%']['GT']        = petro_gt['porosity']['porosity_percent']
    summary['SSA_per_m']['GT']         = petro_gt['ssa']['SSA_per_m']
    summary['Permeability_mD_KC']['GT'] = petro_gt['kc']['K_mD']
if reg_metrics is not None:
    summary['regression_metrics_prob_vs_GT'] = reg_metrics
if seg_m_05 is not None:
    summary['segmentation_metrics'] = {'thr0.5': seg_m_05, 'otsu': seg_m_otsu, 'otsu_threshold': otsu_t}

json_path = os.path.join(cfg.OUT_DIR, 'inference_summary.json')
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"  ✓ {json_path}")


## 15 · Ringkasan

In [ ]:
print("="*65); print("  INFERENCE SUMMARY"); print("="*65)
print(f"  Model        : {os.path.basename(cfg.MODEL_PATH)}")
print(f"  Input        : {os.path.basename(cfg.INPUT_PATH)}  shape={list(prob_vol.shape)}")
print(f"  Output dir   : {cfg.OUT_DIR}")
print()
print(f"  Porositas (thr=0.5) : {petro_05['porosity']['porosity_percent']:.3f} %")
print(f"  Porositas (Otsu)    : {petro_otsu['porosity']['porosity_percent']:.3f} %")
if petro_gt is not None:
    print(f"  Porositas (GT)      : {petro_gt['porosity']['porosity_percent']:.3f} %")
print()
print(f"  Permeabilitas K-C (thr=0.5) : {petro_05['kc']['K_mD']:.3f} mD")
print(f"  Permeabilitas K-C (Otsu)    : {petro_otsu['kc']['K_mD']:.3f} mD")
if petro_gt is not None:
    print(f"  Permeabilitas K-C (GT)      : {petro_gt['kc']['K_mD']:.3f} mD")

if seg_m_05 is not None:
    print()
    print(f"  Dice/IoU (thr=0.5) : {seg_m_05['Dice']:.4f} / {seg_m_05['IoU']:.4f}")
    print(f"  Dice/IoU (Otsu)    : {seg_m_otsu['Dice']:.4f} / {seg_m_otsu['IoU']:.4f}")

print()
print("  Files in output dir:")
for f in sorted(os.listdir(cfg.OUT_DIR)):
    fp = os.path.join(cfg.OUT_DIR, f)
    if os.path.isfile(fp):
        print(f"    {f:35s}  {os.path.getsize(fp)/1e6:>7.2f} MB")
